# Verificação de Idempotência — Camada Bronze

**Tech Challenge Fase 2 — POSTECH AI Scientist**

Valida que o script de preparação da camada **Bronze** pode ser executado múltiplas vezes sem alterar o estado final do data lake — propriedade essencial para reprocessos e retries seguros em produção.

---

**Por que idempotência importa?** Numa pipeline orquestrada (Step Functions, retries automáticos, backfills manuais), qualquer job pode ser executado mais de uma vez sobre o mesmo insumo — por falha transitória, reprocesso ou operação humana. Um job **idempotente** produz exatamente o mesmo estado final do lake em todas as execuções: sem duplicar registros, sem apagar histórico, sem variar conteúdo.

**Como o job Glue garante isso?** Escrita Parquet com `mode("overwrite")` + `partitionBy("ano")` + `spark.sql.sources.partitionOverwriteMode=dynamic`: cada execução sobrescreve **apenas as partições `ano=YYYY` presentes no lote**, deixando as demais intactas.

**Como este notebook verifica?** Usamos a simulação local fiel da pipeline (`pipeline_local.py`, pandas), executamos a camada **duas vezes** e comparamos *snapshots* do lake — uma linha por partição física com contagem de linhas e hash MD5 do conteúdo de negócio (colunas voláteis de timestamp excluídas, pois mudam legitimamente a cada execução). Idempotência confirmada = mesmas partições, mesmas contagens, mesmos hashes.

In [1]:
# Simulação local fiel dos jobs Glue (mesma lógica de transformação,
# DQ e particionamento Hive-style por ano) — ver notebooks/pipeline_local.py
import tempfile
from pathlib import Path

import pandas as pd

from pipeline_local import (
    run_bronze, run_silver, run_gold,
    snapshot_camada, comparar_snapshots,
)

DADOS_DIR = Path('../dados')
LAKE_DIR = Path(tempfile.mkdtemp(prefix='lake_'))
print(f'Data lake local desta verificação: {LAKE_DIR}')

Data lake local desta verificação: /sessions/gallant-youthful-faraday/tmp/lake_jggwmtq7


## Execução 1 — estado inicial da camada Bronze

In [2]:
resultado_exec1 = run_bronze(DADOS_DIR, LAKE_DIR)
print('Registros processados na execução 1:')
resultado_exec1

Registros processados na execução 1:


{'indicador_municipio': 23995,
 'indicador_uf': 145,
 'meta_brasil': 3,
 'meta_uf': 54,
 'meta_municipio': 10704}

In [3]:
def listar_particoes(base):
    """Lista a estrutura física de partições gravada no lake local."""
    for p in sorted((LAKE_DIR / base).rglob('ano=*')):
        n = len(pd.read_parquet(p / 'dados.parquet'))
        print(f'{p.relative_to(LAKE_DIR)}  ({n} linhas)')

listar_particoes('bronze')

bronze/indicador_municipio/ano=2023  (11547 linhas)
bronze/indicador_municipio/ano=2024  (12448 linhas)
bronze/indicador_uf/ano=2023  (70 linhas)
bronze/indicador_uf/ano=2024  (75 linhas)
bronze/meta_brasil/ano=2023  (1 linhas)
bronze/meta_brasil/ano=2024  (1 linhas)
bronze/meta_brasil/ano=2025  (1 linhas)
bronze/meta_municipio/ano=2023  (5352 linhas)
bronze/meta_municipio/ano=2024  (5352 linhas)
bronze/meta_uf/ano=2023  (27 linhas)
bronze/meta_uf/ano=2024  (27 linhas)


In [4]:
snap_exec1 = snapshot_camada(LAKE_DIR, 'bronze')
snap_exec1

,particao,n_linhas,hash_negocio
0,bronze/indicador_municipio/ano=2023,11547,ecfc09eb2f7fc0188233d89adedd60bb
1,bronze/indicador_municipio/ano=2024,12448,7215e05fbcd6618004daea8dddfec41e
2,bronze/indicador_uf/ano=2023,70,2f36a21dea6fd1fcf0c425c489f0721f
3,bronze/indicador_uf/ano=2024,75,2ca0ecda11caa5ad34285f911ed394fa
4,bronze/meta_brasil/ano=2023,1,f3c98db387866c8a351f80cf13694efb
5,bronze/meta_brasil/ano=2024,1,59500ff444c75e1846865ffe21aefbfe
6,bronze/meta_brasil/ano=2025,1,e7c2360fe41abc9b7eaf430fa4f8d024
7,bronze/meta_municipio/ano=2023,5352,0b7d4aadc75a28a3924e2cba7154b228
8,bronze/meta_municipio/ano=2024,5352,6d6d1cc5d38443baa299cea4c6b90158
9,bronze/meta_uf/ano=2023,27,37c819ee73f2aaa3fdd45263d2796a54


## Execução 2 — reprocesso completo da camada Bronze

Repetimos a execução sobre o mesmo insumo, simulando um retry do orquestrador ou um reprocesso manual.

In [5]:
resultado_exec2 = run_bronze(DADOS_DIR, LAKE_DIR)
snap_exec2 = snapshot_camada(LAKE_DIR, 'bronze')
print('Registros processados na execução 2:')
resultado_exec2

Registros processados na execução 2:


{'indicador_municipio': 23995,
 'indicador_uf': 145,
 'meta_brasil': 3,
 'meta_uf': 54,
 'meta_municipio': 10704}

## Comparação dos snapshots

Partição a partição: mesma estrutura, mesma contagem e mesmo hash de conteúdo de negócio.

In [6]:
comparacao = comparar_snapshots(snap_exec1, snap_exec2)
comparacao

,particao,n_linhas_exec1,hash_negocio_exec1,n_linhas_exec2,hash_negocio_exec2,idempotente
0,bronze/indicador_municipio/ano=2023,11547,ecfc09eb2f7fc0188233d89adedd60bb,11547,ecfc09eb2f7fc0188233d89adedd60bb,True
1,bronze/indicador_municipio/ano=2024,12448,7215e05fbcd6618004daea8dddfec41e,12448,7215e05fbcd6618004daea8dddfec41e,True
2,bronze/indicador_uf/ano=2023,70,2f36a21dea6fd1fcf0c425c489f0721f,70,2f36a21dea6fd1fcf0c425c489f0721f,True
3,bronze/indicador_uf/ano=2024,75,2ca0ecda11caa5ad34285f911ed394fa,75,2ca0ecda11caa5ad34285f911ed394fa,True
4,bronze/meta_brasil/ano=2023,1,f3c98db387866c8a351f80cf13694efb,1,f3c98db387866c8a351f80cf13694efb,True
5,bronze/meta_brasil/ano=2024,1,59500ff444c75e1846865ffe21aefbfe,1,59500ff444c75e1846865ffe21aefbfe,True
6,bronze/meta_brasil/ano=2025,1,e7c2360fe41abc9b7eaf430fa4f8d024,1,e7c2360fe41abc9b7eaf430fa4f8d024,True
7,bronze/meta_municipio/ano=2023,5352,0b7d4aadc75a28a3924e2cba7154b228,5352,0b7d4aadc75a28a3924e2cba7154b228,True
8,bronze/meta_municipio/ano=2024,5352,6d6d1cc5d38443baa299cea4c6b90158,5352,6d6d1cc5d38443baa299cea4c6b90158,True
9,bronze/meta_uf/ano=2023,27,37c819ee73f2aaa3fdd45263d2796a54,27,37c819ee73f2aaa3fdd45263d2796a54,True


In [7]:
# Validação formal: qualquer divergência interrompe o notebook com erro
assert comparacao['idempotente'].all(), 'FALHA: execuções produziram estados diferentes!'
assert len(snap_exec1) == len(snap_exec2), 'FALHA: número de partições divergente!'

print('=' * 60)
print(f'  IDEMPOTÊNCIA CONFIRMADA — {len(comparacao)} partições idênticas')
print(f'  nas duas execuções (estrutura, contagens e conteúdo).')
print('=' * 60)

  IDEMPOTÊNCIA CONFIRMADA — 11 partições idênticas
  nas duas execuções (estrutura, contagens e conteúdo).


## Conclusão

A camada **Bronze** é idempotente: reprocessar a ingestão dos CSVs produz exatamente as mesmas partições `bronze/<entidade>/ano=YYYY/`, com as mesmas contagens e o mesmo conteúdo de negócio (incluindo o `_record_hash` MD5, que é determinístico por construção — função apenas da chave composta do registro).

No ambiente AWS, essa propriedade é garantida pelo `etl_bronze.py` com `mode("overwrite")` + `partitionBy("ano")` + `partitionOverwriteMode=dynamic`: um retry da ingestão sobrescreve apenas os anos presentes no lote, preservando o histórico dos demais ciclos avaliativos — requisito central para um indicador que exige comparação entre anos.